# Tissue extractor tutorial (not this)

this is really running the regression on subjects; starting with getting normalization files from SUITPy normalization with respect to MNI Symmetric template, then running the linear regression (on white matter for now; June 10, 2026, 12:35 pm).

# should rename this - this isn't the whole tutorial.

Make this file the file for processing (norm - call once, then convert cell to `raw`; reslice - for all images) for slope, intercept images.

## to do for tutorial

- make the base loop a helper function that you can call for each function

- run each function in its own cell using the helper function

- Above is for the multiple subject case

- Also just show how to do it on one subject

In [1]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
import ants

import SUITPy as suit
import SUITPy.atlas as atlas

import nitools as nt
import tissue_extractor as te

from pathlib import Path
import os

In [2]:
# if not in same directory as fcn (e.g. avg_vol.py), import cannot find it since notebook is not in root directory of project.
# so add project root
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

from image_analysis import avg_vol as av

In [4]:
!pwd

/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/image_processing


# BEFORE RUNNING: UPLOAD TO GITHUB REPO (THIS AND TISSUE EXTRACTOR, UPDATED); THEN RUN

"""
dummy function to test loop; to be replaced with the actual function
"""
def dummy_fcn(subj_id, week):
    print(subj_id, week)

In [3]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [4]:
tissue = 'wm' # default
tissue_dict = {
    'gm': 'c1',
    'wm': 'c2',
    'csf': 'c3'
}

In [7]:
te.normalize?

Signature: te.normalize(t1_path, mask_path, results_path, space='SUIT')
Docstring:
June 9: updated with updated suit function
with correct input arg names
and option to choose template space to normalize to
File:      ~/Documents/GitHub/smarts_cerebellum/image_processing/tissue_extractor.py
Type:      function

subj_id = 'CU_2310'
week = 'W0'

results_path = Path(anat_dir)/subj_id/week/'iso_norm_mniSymm/' # folder specifies space

t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'


# ask joern
before running this!
In the regression module, should I put it back to voxel coordinates before writing image (intercept and slope)?

Also should talk to Joern about the voxel reg on just the t1 anatomical.

Also ask Joern about the function - is it correct for aligning voxels for other weeks (is in voxel coords now?); also read the function and figure it out dummy

# check the code for the new week_path before running

# restart kernel for updated av function before calling this

figure out why it cannot find image_analysis folder; as a temporary solution, just write the function here.

In [5]:
p_df=p_df[p_df.subj_id == 'CU_2538']

In [7]:
p_df

,SN,ID,Centre,Week,week,RefT1,numrun,nslices,Hand,LesionSide,...,surfmvpa,include1,lesiondef,behavior_missing,behavior_blocks,has_mvc,DTImap_missing,CentreNo,machine,subj_id
5,6,2538,CU,W0,0,W0,8,35,b,right,...,1,1,1,0,8,1,0,1,naveed,CU_2538
6,7,2538,CU,W4,4,W0,8,35,b,right,...,1,1,1,0,8,1,0,1,naveed,CU_2538


In [ ]:
# CALL FOR WM SEGMENTATION IMAGE IN NATIVE SPACE________change image suffix for other types

betas = [] # store matrices for all subjects

for subj in p_df['subj_id'].unique():
    #betas.append(avg_vol(subj, ref_img))

    # make directory to store output for each subj
    results_path = Path(anat_dir)/subj/'regression_native/'
    #results_path = f'{anat_dir}/{subj}/regression_native/'
    results_path.mkdir(parents=True, exist_ok = True) # exist_ok = True

    # progress check: note what image is being used
    if tissue:
        print(tissue)

    # fix: use tissue_dict for ref_img
    refT1 = (p_df.loc[(p_df['subj_id']==subj), 'RefT1'].iloc[0]).strip()
    ref_img = f'{anat_dir}/{subj}/{refT1}/{tissue_dict[tissue]}{subj}_{refT1}_T1.nii' # should specify tissue in this call for reference iamge

    betas.append(av.avg_vol(subj, 
            reference_img=ref_img, # reference anatomical
            #week_path, # path to week image
            results_path = results_path, # maybe specify folder for this, too (e.g. native_regression)
            image_suffix = f'{tissue}_native', # if using tissue; otherwise, change for T1 anat
            tissue=f'{tissue}'
            ))
    
"""
    betas.append(av.avg_vol(subj_id = subj, reference_img=ref_img,
                         results_path = f'{anat_dir}/{subj}/'),
                         image_suffix = "wm_native"
                         )
                         """
    
    # we don't need this weeks loop, the function does it. Really just need to loop through the subject and get their reference img
    # even better if the function can get the reference image itself.
    
    #week = p_df.loc[(p_df['subj_id']==subj), 'week'].iloc[1]
   



on week 0 for CU_2538
on week 4 for CU_2538


'\n    betas.append(av.avg_vol(subj_id = subj, reference_img=ref_img,\n                         results_path = f\'{anat_dir}/{subj}/\'),\n                         image_suffix = "wm_native"\n                         )\n                         '

In [10]:
tissue

'wm'

# note
In the present avg_vol function, week paths are for c2; this is terrible, need to fix (i.e. have a tissue_dict, where if tissue = None, then it'll just put nothing in front, so like):

tissue_dict = {
    'gm': c1,
    'wm': c2,
    'csf': c3,
    None: '' # i don't know if this notation would work
}

Otherwise, we can just have an if tissue!=None statement, so if tissue is supplied, it'll use the dict above (minus None coding, which is strange and might not work) to find the correct prefix of the file.

I did the latter.

Should check that this works well on one subject before continuing wiht the rest. ANd need to ask Joern about the voxel coordiantes thing first, but also check the code from nilearn.